# AnMi server trên Kaggle

Chạy backend AnMi (CogAI) cho robot, sau tên miền ngrok cố định. Robot dùng OTA URL `https://<NGROK_DOMAIN>/xiaozhi/ota/`.

**Chuẩn bị (một lần):**
1. *Add Input* → dataset tải lên từ `anmi-backend.zip` (tạo bằng `cogai/deploy/kaggle/make_bundle.py`; Kaggle tự giải nén cũng được).
2. *Add-ons → Secrets*, bật cho notebook này: `OPENAI_API_KEY`, `NGROK_AUTHTOKEN`, `NGROK_DOMAIN`, `ANMI_DEVICE_TOKENS` (tùy chọn `AZURE_SPEECH_KEY`).
3. *Settings*: **Internet on**. Không cần GPU.
4. Chạy lâu dài: **Save Version → Save & Run All (Commit)**; phiên chạy tối đa 12 giờ rồi dừng.

Không ghi key vào notebook. Robot giữ hội thoại trong bộ nhớ phiên, không lưu xuống đĩa; mọi thứ mất khi phiên Kaggle kết thúc.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def secret(name, required=True):
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = ''
    if required and not value:
        raise SystemExit(f'Thiếu Kaggle Secret: {name} (Add-ons → Secrets)')
    return value

OPENAI_API_KEY = secret('OPENAI_API_KEY')
NGROK_AUTHTOKEN = secret('NGROK_AUTHTOKEN')
NGROK_DOMAIN = secret('NGROK_DOMAIN').removeprefix('https://').strip('/')
ANMI_DEVICE_TOKENS = secret('ANMI_DEVICE_TOKENS')
AZURE_SPEECH_KEY = secret('AZURE_SPEECH_KEY', required=False)
print('Đã đọc secrets. Tên miền:', NGROK_DOMAIN)

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

APP = '/kaggle/working/anmi'
shutil.rmtree(APP, ignore_errors=True)
bundles = glob.glob('/kaggle/input/**/anmi-backend.zip', recursive=True)
# Kaggle unpacks an uploaded .zip into the dataset, so look for the unpacked backend too.
unpacked = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/requirements.lock.txt', recursive=True)
            if os.path.exists(os.path.join(os.path.dirname(p), 'app', 'main.py'))]
if bundles:
    zipfile.ZipFile(bundles[0]).extractall(APP)
elif unpacked:
    shutil.copytree(unpacked[0], APP)
else:
    raise SystemExit('Không thấy mã nguồn AnMi: Add Input dataset anmi-backend.')
print('Mã nguồn từ:', bundles[0] if bundles else unpacked[0])
# The backend needs Python 3.12; uv installs it alongside Kaggle's own Python.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv', 'pyngrok'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '-q', '--python', '3.12', f'{APP}/.venv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '-q', '--python', f'{APP}/.venv/bin/python',
                '-r', f'{APP}/requirements.lock.txt'], check=True)
print('Đã cài backend AnMi.')

In [ ]:
import time, urllib.request

# Same live settings as `node cogai/scripts/dev.mjs --live --device`.
env = {**os.environ,
       'COGAI_MODE': 'live', 'COGAI_LLM_API_KEY': OPENAI_API_KEY,
       'COGAI_LLM_BASE_URL': 'https://api.openai.com/v1', 'COGAI_LLM_MODEL': 'gpt-4.1-mini',
       'COGAI_TRANSCRIPTION_MODEL': 'gpt-4o-transcribe',
       'COGAI_SPEECH_PROVIDER': 'auto', 'COGAI_EDGE_VOICE': 'vi-VN-HoaiMyNeural', 'COGAI_EDGE_RATE': '+0%',
       'COGAI_AZURE_SPEECH_KEY': AZURE_SPEECH_KEY, 'COGAI_AZURE_SPEECH_REGION': 'southeastasia',
       'COGAI_LLM_MAX_COMPLETION_TOKENS': '1800', 'COGAI_EXTERNAL_DAILY_CALL_LIMIT': '0',
       'COGAI_RESEARCH_DEEP_ENABLED': 'true', 'COGAI_EMBEDDING_MODEL': 'text-embedding-3-large',
       'COGAI_KNOWLEDGE_DEVELOPMENT': 'true', 'COGAI_VOICE_MAX_RECORDING_SECONDS': '45',
       'COGAI_DEVICE_GATEWAY': 'true', 'COGAI_DEVICE_TOKENS': ANMI_DEVICE_TOKENS,
       'COGAI_DATA_DIR': f'{APP}/data'}
log = open('/kaggle/working/anmi.log', 'w')
server = subprocess.Popen([f'{APP}/.venv/bin/python', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1',
                           '--port', '8000', '--no-access-log'], cwd=APP, env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2).read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit('Server không khởi động được, xem /kaggle/working/anmi.log')

In [ ]:
"""Paste this cell after the backend startup cell in the Kaggle notebook."""
import json
import re
import urllib.error
import urllib.request
from urllib.parse import urlsplit

from pyngrok import conf, ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

domain = NGROK_DOMAIN.strip().removeprefix('https://').rstrip('/')
if not domain or any(char in domain for char in '/?#@'):
    raise RuntimeError('NGROK_DOMAIN chỉ được chứa tên miền, không kèm đường dẫn OTA.')

macs = []
for entry in ANMI_DEVICE_TOKENS.strip().split(';'):
    mac, separator, token = entry.strip().partition('=')
    mac = mac.lower().replace('-', ':').strip()
    if not separator or not token.strip() or not re.fullmatch(r'[0-9a-f]{2}(:[0-9a-f]{2}){5}', mac):
        raise RuntimeError('ANMI_DEVICE_TOKENS phải có dạng MAC=token;MAC=token, trên một dòng.')
    if mac in macs:
        raise RuntimeError(f'MAC bị lặp trong Secret: {mac}')
    macs.append(mac)


def check_ota(origin, mac, location):
    request = urllib.request.Request(
        origin + '/xiaozhi/ota/', data=b'{}', method='POST',
        headers={'Device-Id': mac, 'Content-Type': 'application/json',
                 'User-Agent': 'AnMi-Notebook/1.0', 'ngrok-skip-browser-warning': '1'})
    try:
        with urllib.request.urlopen(request, timeout=15) as response:
            answer = json.load(response)
    except urllib.error.HTTPError as error:
        if error.code == 404:
            detail = ('Backend thiếu route OTA: kiểm tra COGAI_DEVICE_GATEWAY=true và gói backend.'
                      if location == 'local' else
                      'Backend nội bộ đã đạt nhưng URL công khai trả 404: kiểm tra tunnel/domain ngrok.')
        elif error.code == 403:
            detail = f'Backend chưa chấp nhận MAC {mac}. Chạy lại cell Secrets và khởi động lại backend.'
        else:
            detail = 'Kiểm tra notebook backend và trạng thái tunnel ngrok.'
        raise RuntimeError(f'{location}: HTTP {error.code}. {detail}') from None
    except (urllib.error.URLError, TimeoutError):
        raise RuntimeError(f'{location}: không kết nối được. Kiểm tra backend, mạng và tunnel.') from None
    except ValueError:
        raise RuntimeError(f'{location}: phản hồi không phải JSON của AnMi.') from None
    websocket = answer.get('websocket', {})
    if not websocket.get('url') or not websocket.get('token'):
        raise RuntimeError(f'{location}: phản hồi OTA thiếu cấu hình WebSocket.')
    if location == 'public':
        endpoint = urlsplit(websocket['url'])
        if endpoint.scheme != 'wss' or endpoint.netloc != domain or endpoint.path != '/xiaozhi/v1/':
            raise RuntimeError('OTA trả URL WebSocket sai host/giao thức. Kiểm tra proxy HTTPS của backend.')
    return websocket['url']  # Never print or save the token.


# Verify the actual local gateway before exposing or diagnosing a tunnel.
for mac in macs:
    check_ota('http://127.0.0.1:8000', mac, 'local')
    print('Backend nội bộ OK:', mac)

conf.get_default().auth_token = NGROK_AUTHTOKEN
public_origin = f'https://{domain}'
tunnel = next((item for item in ngrok.get_tunnels()
               if item.public_url.rstrip('/') == public_origin), None)
if tunnel is None:
    try:
        tunnel = ngrok.connect(8000, 'http', url=public_origin)
    except PyngrokNgrokHTTPError:
        tunnel = ngrok.connect(8000, 'http', domain=domain)

OTA_URL = public_origin + '/xiaozhi/ota/'
print('OTA URL cho cả hai robot:', OTA_URL)
for mac in macs:
    websocket_url = check_ota(public_origin, mac, 'public')
    print('OTA công khai HTTP 200:', mac, '| WebSocket:', websocket_url)


In [ ]:
# Keep the session alive and report health every 10 minutes (Kaggle stops it at 12 hours).
started = time.time()
while time.time() - started < 11.9 * 3600:
    if server.poll() is not None:
        raise SystemExit('Server đã dừng, xem /kaggle/working/anmi.log')
    turns = [line.strip() for line in open('/kaggle/working/anmi.log', encoding='utf-8', errors='ignore')
             if 'device turn' in line]
    print(time.strftime('%H:%M'), 'AnMi đang chạy |', len(turns), 'lượt nói |', turns[-1] if turns else '')
    time.sleep(600)